# Bab 1: Common  Conventions and API Elements of scikit-learn

*Notebook ringkasan **scikit-learn Cookbook, Edisi Ketiga** (John Sukup, Packt 2025) — Bab 1.*

Bab ini tidak membahas pemodelan yang rumit. Tujuannya adalah mengenalkan **istilah dan konvensi** yang digunakan pada bab-bab berikutnya: *estimator*, *transformers*, *pipeline*, serta metode `fit()`, `predict()`, dan `transform()`.

**Gambaran singkat bab**

| Topik | Gagasan utama |
|:--|:--|
| Filosofi desain | Konsistensi, kesederhanaan, modularitas, penggunaan ulang |
| Estimator | Objek yang mempelajari data melalui `fit()` |
| Transformer | Estimator yang mengubah data melalui `transform()` |
| Estimator kustom | Subkelas `BaseEstimator` + mixin |
| Pipeline | Menggabungkan beberapa langkah dalam satu objek; mencegah kebocoran data |
| Atribut dan metode | `coef_`, `intercept_`, `score()` |
| Penyetelan hyperparameter | `get_params()`, `set_params()`, pencarian grid/acak |
| Metadata | Tag estimator dan perutean metadata |
| Praktik terbaik | API seragam, prapemrosesan, pipeline, validasi silang, penyetelan |

In [1]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"NumPy version: {np.__version__}")

scikit-learn version: 1.9.1
NumPy version: 2.5.3


> **Catatan versi.** Buku ini menargetkan scikit-learn **1.5+**. Kode di bawah dapat dijalankan pada versi terbaru; versi yang ditampilkan di atas adalah versi yang terpasang pada sesi ini. Beberapa detail API kecil (misalnya `multi_class` pada `LogisticRegression` yang digunakan di Bab 6) berubah pada rilis yang lebih baru. Perubahan tersebut akan diberi catatan pada bagian terkait.

## 1. scikit-learn's Design Philosophy

### Summary
The library is built on four principles:

1. **Konsistensi** – setiap algoritma menyediakan sekumpulan metode yang sama (`fit`, `predict`, `transform`).
2. **Kesederhanaan** – nilai bawaan hyperparameter yang masuk akal memungkinkan pengguna memperoleh hasil dengan cepat.
3. **Modularitas** – estimator, , dan pipeline merupakan komponen mandiri yang dapat dirangkai.
4. **Penggunaan ulang** – komponen yang ditulis sekali, termasuk komponen buatan sendiri, dapat digunakan dalam alur kerja lain.

Akibatnya, mengganti `LogisticRegression()` dengan `RandomForestClassifier()` hanya membutuhkan **perubahan satu baris**.

> *Catatan nama:* penulisan yang benar adalah **scikit-learn** dengan huruf kecil (diucapkan "sy-kit"); *sci* berasal dari kata *science*.

### Penjelasan teori
**Mengapa API yang seragam penting?** Dalam desain perangkat lunak, scikit-learn menerapkan gagasan *interface segregation / duck typing*: objek apa pun yang memiliki `fit` (serta `predict` atau `transform`) dapat digunakan dengan alat scikit-learn seperti `Pipeline`, `GridSearchCV`, dan `cross_val_score`. Alat tersebut tidak perlu mengetahui algoritma yang digunakan, cukup memastikan objek itu memiliki perilaku seperti estimator. Karena itu, validasi silang, penyetelan, dan pipeline dapat digunakan dengan lebih dari 100 algoritma tanpa penanganan khusus.

## 2. Understanding Estimators

### Summary
**Estimator** adalah objek yang mempelajari data. Metode utamanya adalah `fit()`. Estimator untuk pembelajaran terawasi juga menyediakan `predict()`.

Pola umum tiga langkah:
```python
model = Estimator(hyperparameters)   # 1. create
model.fit(X_train, y_train)          # 2. learn
model.predict(X_new)                 # 3. use
```

### Penjelasan teori — Linear Regression
Diberikan sebuah matriks fitur $\mathbf{X}\in\mathbb{R}^{n\times p}$ (n samples, p features) and target $\mathbf{y}\in\mathbb{R}^n$, linear regression assumes

$$\hat y = \mathbf{w}^T\mathbf{x} + b$$

dan memilih $\mathbf{w}, b$ untuk meminimalkan **jumlah kuadrat residual** (*Ordinary Least Squares*, OLS):

$$\min_{\mathbf{w},b}\sum_{i=1}^{n}\bigl(y_i-(\mathbf{w}^T\mathbf{x}_i+b)\bigr)^2$$

Solusi bentuk tertutupnya disebut **persamaan normal** (intersep dimasukkan ke dalam $\mathbf{X}$ sebagai kolom berisi angka satu):

$$\mathbf{w}=(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

*Intuisi:* kita memilih garis yang jumlah kuadrat jarak vertikalnya terhadap seluruh titik sekecil mungkin.

In [2]:
from sklearn.linear_model import LinearRegression

X = np.array([[1], [2], [3], [4], [5]])   # matriks fitur (5 sampel, 1 fitur)
y = np.array([1, 2, 3, 3.5, 5])           # nilai target

model = LinearRegression()
model.fit(X, y)

X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print("Prediksi untuk X = [6, 7]:", predictions)

Predictions for X = [6, 7]: [5.75 6.7 ]


**Membaca keluaran.** Hasil prediksi adalah $5.75$ untuk $x=6$ and $6.70$ untuk $x=7$. Hasil tersebut berasal dari garis yang dipelajari $\hat y = 0.95x+0.05$ (koefisiennya diperiksa pada bagian 6): $0.95\times6+0.05=5.75$.

### `fit_predict()` — shortcut untuk unsupervised learning

`fit_predict()` = `fit()` + `predict()` on the **same data**. It is mainly used untuk **clustering**, dengan there is no target $\mathbf{y}$ and you want the labels of the training data itself.

**Teori K-Means.** K-Means membagi $n$ titik menjadi $K$ cluster dengan meminimalkan **jumlah kuadrat dalam cluster** (*denganin-cluster sum of squares*, WCSS, disebut juga *inertia*):

$$\min_{\{C_k\}}\sum_{k=1}^{K}\sum_{\mathbf{x}_i\in C_k}\lVert\mathbf{x}_i-\boldsymbol\mu_k\rVert^2,\qquad \boldsymbol\mu_k=\frac1{|C_k|}\sum_{\mathbf{x}_i\in C_k}\mathbf{x}_i$$

Algoritma (Lloyd): (1) inisialisasi K centroid → (2) tempatkan setiap titik ke centroid terdekat → (3) pindahkan setiap centroid ke nilai rata-rata titik dalam kelompoknya → ulangi langkah 2–3 sampai tidak ada perubahan.

In [3]:
from sklearn.cluster import KMeans

X = np.array([[1], [2], [3], [4], [5]])

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
print("Label cluster:", labels)
print("Pusat cluster:", kmeans.cluster_centers_.flatten())

Cluster labels: [0 0 0 1 1]
Cluster centers: [2.  4.5]


**Membaca keluaran.** Titik terbagi menjadi dua kelompok dan setiap pusat merupakan nilai rata-rata kelompoknya. Label 0 atau 1 diberikan secara arbitrer; yang penting adalah hasil pengelompokannya.

**Kapan masing-masing digunakan**
- *Pembelajaran terawasi:* jalankan `fit(X_train, y_train)`, lalu `predict(X_test)` secara terpisah. Data latih dan data uji harus tetap terpisah.
- *Unsupervised:* `fit_predict(X)` is convenient when you want labels untuk the training data itself. Train/test splits are still possible if you want to test generalisation.

## 3. Transformers and `Transform()`

### Summary
A transformer is an estimator that *modifies data* instead of predicting. Methods:

- `fit(X)` – mempelajari parameter dari data (misalnya rata-rata dan standar deviasi)
- `transform(X)` – menerapkan hasil yang telah dipelajari
- `fit_transform(X)` – melakukan keduanya dalam satu langkah

### Penjelasan teori — Standardization (z-score)
$$z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}$$

Setiap fitur $j$ diubah agar memiliki rata-rata 0 dan standar deviasi 1. **Mengapa?** Algoritma berbasis jarak dan gradien (KNN, SVM, regresi logistik, jaringan saraf, PCA) sensitif terhadap skala fitur. Fitur bernilai ribuan, seperti pendapatan, dapat mendominasi fitur bernilai desimal, seperti IPK. Standardisasi membuat skala fitur lebih sebanding.

> Catatan: `StandardScaler` menggunakan standar deviasi **populasi** (dibagi $n$, bukan $n-1$).

In [4]:
from sklearn.preprocessing import StandardScaler

X = np.array([[1, 2], [3, 4], [5, 6]])

scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)

print("Data asli:\n", X)
print("\nData setelah penskalaan:\n", X_scaled)
print("\nRata-rata setiap fitur (hasil pembelajaran):", scaler.mean_)
print("Standar deviasi setiap fitur (hasil pembelajaran):", scaler.scale_)

Original data:
 [[1 2]
 [3 4]
 [5 6]]

Scaled data:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Mean of each feature (learned): [3. 4.]
Std of each feature (learned): [1.63299316 1.63299316]


**Pemeriksaan manual:** $z_{11}=\dfrac{1-3}{1.633}=-1.2247$.

### Aturan penting: hindari kebocoran data
- `fit_transform()` hanya pada data **latih**.
- `transform()` (jangan gunakan `fit`) pada data **uji**.

Re-fitting on test data computes a *different* mean/std, so train and test would be on different scales — and, worse, test inuntukmation leaks into preprocessing.

In [5]:
X_scaled = StandardScaler().fit_transform(X)
print(X_scaled)
print("Rata-rata kolom:", X_scaled.mean(axis=0))
print("Standar deviasi kolom:", X_scaled.std(axis=0))

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
Column means: [0. 0.]
Column stds : [1. 1.]


## 4. Custom Estimators and Transformers

### Summary
Kamu dapat membuat komponen sendiri dengan membuat subkelas dari `BaseEstimator` dan menambahkan **mixin**:

| Mixin | Fungsi tambahan |
|:--|:--|
| `ClassifierMixin` | `score()` = akurasi |
| `RegressorMixin` | `score()` = $R^2$ |
| `Mixin` | `fit_transform()` |

### Penjelasan teori — mixins
A **mixin** is a small class that supplies *one* capability and is combined dengan others through multiple inheritance, instead of building a deep class tree. `Mixin` simply implements `fit_transform(X) = self.fit(X).transform(X)` — which is why you only need to write `fit` and `transform`.

**Konvensi yang perlu diikuti setiap estimator kustom**
1. `__init__` hanya *menyimpan* hyperparameter (tanpa perhitungan) → memungkinkan penggunaan `get_params()/set_params()` and `clone()`.
2. `fit()` mengembalikan `self` → memungkinkan perangkaian.
3. Atribut hasil pembelajaran diakhiri garis bawah (`n_features_in_`).
4. `check_is_fitted()` guards `transform/predict`.

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class ValueClipper(BaseEstimator, TransformerMixin):
    "Clips all feature values to the [lower, upper] range."

    def __init__(self, lower=0.0, upper=1.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.n_features_in_ = np.asarray(X).shape[1]
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_is_fitted(self, 'is_fitted_')
        return np.clip(np.asarray(X), self.lower, self.upper)

X = np.array([[0.5, -0.3], [1.5, 0.7], [0.2, 2.1]])
clipper = ValueClipper(lower=0.0, upper=1.0)
print(clipper.fit_transform(X))      # fit_transform tersedia dari TransformerMixin
print("Parameter:", clipper.get_params())

[[0.5 0. ]
 [1.  0.7]
 [0.2 1. ]]
Parameters: {'lower': 0.0, 'upper': 1.0}


Nilai di luar $[0,1]$ dipindahkan ke batas terdekat ($-0.3\to0$, $1.5\to1$, $2.1\to1$). Karena `ValueClipper` mengikuti konvensi tersebut, komponen ini dapat langsung digunakan dalam `Pipeline` atau `GridSearchCV` (untuk menyetel `lower`/`upper`).

## 5. Pipelines and Workflow Automation

### Summary
A **Pipeline** chains s and (optionally) a final estimator into one object. `fit()` runs `fit_transform()` on each  then `fit()` on the last step; `predict()`/`score()` run only `transform()` on the s.

### Penjelasan teori
For s $T_1,\dots,T_K$ and final estimator $E$:

$$\hat y=E\bigl(T_K(\dots T_2(T_1(\mathbf X)))\bigr)$$

Manfaatnya: **tidak ada kebocoran data** (prapemrosesan dipelajari hanya dari data yang diberikan ke `fit`), **hasil yang dapat direproduksi**, **satu artefak yang siap diterapkan**, dan seluruh alur dapat disetel menggunakan `GridSearchCV`.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=200, n_features=5, n_informative=3,
                           n_redundant=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
pipe.fit(X_train, y_train)

print(f"Akurasi data latih: {pipe.score(X_train, y_train):.4f}")
print(f"Akurasi data uji:      {pipe.score(X_test, y_test):.4f}")
print("Pipeline steps:   ", [name for name, _ in pipe.steps])

Training accuracy: 0.9267
Test accuracy:     0.8800
Pipeline steps:    ['scaler', 'classifier']


**Membaca keluaran.** Akurasi data latih dan data uji berdekatan, sehingga model tidak mengalami overfitting yang parah (selisih merupakan perbedaan akurasi dalam bentuk proporsi, bukan "poin persentase").

### Catatan MLOps
Production ML is a cycle, not a line: **Continuous Training (CT)**, **CI/CD**, and **monitoring** (drift detection). *"There is no such thing as a model that works well untukever."* A `Pipeline` is the natural unit to version and deploy (e.g. dengan `joblib`, MLflow, Kubeflow) — Chapter 13 covers this.

## 6. Common Attributes and Methods

### Summary
Nilai hasil pembelajaran disimpan dalam atribut yang diakhiri **garis bawah** (hanya tersedia setelah `fit`): `coef_`, `intercept_`, `feature_importances_`, `cluster_centers_`, …

`score()` returns a default metric: **accuracy** untuk classifiers, **$R^2$** untuk regressors.

### Penjelasan teori — $R^2$
$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}=1-\frac{SS_{res}}{SS_{tot}}$$
Nilai ini menunjukkan proporsi variasi target yang dapat dijelaskan model (1 = sempurna, 0 = tidak lebih baik daripada memprediksi rata-rata). Perlu diperhatikan: nilainya **tidak pernah turun saat fitur ditambahkan**, bahkan jika fitur tersebut hanya berupa noise. Versi **adjusted** memberikan penalti untuk hal ini:

$$R^2_{adj}=1-\frac{(1-R^2)(n-1)}{n-p-1}$$

In [8]:
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression().fit(X, y)
print("Koefisien (w):", model.coef_)
print("Intersep (b):      ", model.intercept_)
print(f"R-kuadrat:          {model.score(X, y):.6f}")

y_hat = model.coef_[0] * 3 + model.intercept_
print(f"\nPrediksi manual untuk X=3: {y_hat:.2f}   (actual y = {y[2]})")

Coefficients (w): [0.95]
Intercept (b):    0.04999999999999938
R-squared:         0.980978

Manual prediction for X=3: 2.90   (actual y = 3.0)


Garis hasil pemodelan adalah $\hat y=0.95x+0.05$ dengan $R^2\approx0.981$: sekitar 98% variasi dapat dijelaskan. Nilainya tidak mencapai 1,0 karena titik $(4,\,3.5)$ berada di bawah tren garis lurus.

## 7. Hyperparameter Tuning with Search Methods

### Summary
**Parameters** are learned during `fit` (e.g. `coef_`). **Hyperparameters** are chosen *beuntuke* training and tell the model *how to learn* (e.g. `n_estimators`, `C`, `max_depth`).

Ada dua cara untuk mengaturnya:
1. Secara manual: `set_params()` / `get_params()`.
2. Secara otomatis: `GridSearchCV` (menyeluruh), `RandomizedSearchCV` (sampel acak), plus successive-halving variants (`HalvingGridSearchCV`, `HalvingRandomSearchCV`).

In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.set_params(n_estimators=100, max_depth=10, random_state=42)

for key, value in sorted(model.get_params().items()):
    print(f"  {key}: {value}")

  bootstrap: True
  ccp_alpha: 0.0
  class_weight: None
  criterion: gini
  max_depth: 10
  max_features: sqrt
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 1
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 100
  n_jobs: None
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False


### Penjelasan teori — key Random Forest hyperparameters
- `n_estimators` – jumlah pohon. Semakin banyak pohon → varians semakin rendah, tetapi biaya komputasi meningkat secara linear.
- `max_depth` – kedalaman maksimum pohon; membatasi kompleksitas (regularisasi).
- `max_features='sqrt'` – setiap pemisahan hanya mempertimbangkan $\sqrt p$ fitur acak, yang **mengurangi korelasi** antarpohon (sumber penurunan varians pada ensemble).
- `criterion='gini'` – kualitas pemisahan. **Impuritas Gini**:

$$G=1-\sum_{c=1}^{C}p_c^2$$

dengan $p_c$ merupakan proporsi kelas $c$ di dalam node. Node murni → $G=0$; node biner dengan pembagian 50/50 → $G=0.5$.
- `random_state` – reprodusibilitas.

## 8. Metadata: Tags and Metadata Routing

### Summary
- **Tag estimator** menjelaskan kemampuan estimator (mendukung banyak keluaran? menangani NaN? membutuhkan masukan positif?). Infrastruktur scikit-learn membaca tag tersebut sehingga tidak perlu mengandalkan percobaan berulang.
- **Metadata routing** controls how extra per-sample inuntukmation (`sample_weight`, `groups`, …) flows through meta-estimators.
  - **Routers** (`Pipeline`, `GridSearchCV`) untukward metadata.
  - **Konsumen** (estimator, scorer) menggunakan metadata tersebut.

*Contoh:* pada masalah dengan distribusi kelas yang tidak seimbang, arahkan `sample_weight` ke langkah classifier, tetapi tidak ke scaler.

In [10]:
lr = LogisticRegression()
tags = lr.__sklearn_tags__()
print("Estimator:", type(lr).__name__)
print("jenis_estimator:", tags.estimator_type)
print("non_deterministik:", tags.non_deterministic)
print("dukungan_array_api:", tags.array_api_support)

Estimator: LogisticRegression
estimator_type: classifier
non_deterministic: False
array_api_support: True


Tag berfungsi sebagai **kontrak** antara estimator dan mekanisme scikit-learn. Kamu terutama akan menggunakannya saat menulis estimator kustom.

## 9. Best Practices for API Usage

1. **Uniuntukm API** – learn `fit → predict/transform` once.
2. **Preprocess** (scale, encode, impute) beuntuke modelling.
3. **Use pipelines** untuk multi-step workflows (no leakage, reproducible).
4. **Lakukan validasi silang** – jangan menilai model hanya berdasarkan data latih.
5. **Setel hyperparameter** secara sistematis (`GridSearchCV` / `RandomizedSearchCV`).
6. **Periksa bagian internal model** (`coef_`, `feature_importances_`) untuk interpretability.

---
## Ringkasan Chapter 1

| Konsep | Metode utama | Fungsi |
|:--|:--|:--|
| Estimator | `fit()`, `predict()` | Belajar dan memprediksi |
| Transformer | `fit()`, `transform()`, `fit_transform()` | Modify data |
| Pipeline | `fit()`, `predict()`, `score()` | Merangkai langkah |
| Atribut | `coef_`, `intercept_` | Memeriksa hasil pembelajaran |
| Evaluasi | `score()` | Metrik cepat |
| Hyperparameter | `get_params()`, `set_params()` | Mengatur konfigurasi |
| Metadata | `__sklearn_tags__()` | Menjelaskan kemampuan |